# MAPPO Curriculum

This notebook trains the first curriculum stage for `AntByteForagingEnv`: ants learn to reach cookie sources, pick up bites, return to the hub, and write configurable tile values into the environment. The trainer predicts both movement and write-value actions for every ant; `WRITE_BITS = 1` is the current curriculum setting and can be raised to 3, 5, or up to 8 later. Training uses TorchRL's MAPPO loss and multi-agent GAE while keeping local actor observations and a centralized critic.

The training cell below grows the map progressively. It pads observations to the largest scheduled map, so the same actor/critic checkpoint can continue from smaller maps to larger maps. Each episode randomizes the colony location and uses multiple random cookie source locations.

Install the notebook/training extras from the repo root if needed:

```bash
python -m pip install -e ".[rl,notebooks]"
```

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import sys

try:
    import tensordict  # noqa: F401
    import torchrl  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "torchrl/tensordict"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the training extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[rl,notebooks]"'
    ) from exc


In [ ]:
import importlib
from types import SimpleNamespace

import imageio.v2 as imageio
import torch
from tqdm.auto import tqdm

from ant_byte_env import AntByteForagingEnv
from ant_byte_env.vault import create_vault_entry
import train_mappo

importlib.reload(train_mappo)
from train_mappo import (
    MAPPOAgent,
    build_actor_observations,
    build_central_observations,
    build_curriculum_reset_options,
    draw_vision_squares,
    flatten_agent_actions,
    main,
    obs_to_tensor,
    write_value_count,
)


## Quick Smoke Run

Run this first to make sure the notebook kernel can import the repo, create the environment, and complete one tiny TorchRL-backed MAPPO update.

In [ ]:
smoke_metrics = main(
    [
        "--total-timesteps", "8",
        "--num-envs", "1",
        "--num-steps", "4",
        "--num-minibatches", "1",
        "--update-epochs", "1",
        "--width", "4",
        "--height", "4",
        "--num-ants", "1",
        "--food-count", "1",
        "--max-steps", "8",
        "--write-bits", "1",
        "--hidden-size", "16",
        "--seed", "11",
        "--no-cuda",
        "--quiet",
    ]
)
smoke_metrics

## Progressive Map Curriculum

Each stage resumes from the previous stage checkpoint. Keep `--obs-width` and `--obs-height` equal to the largest scheduled map, otherwise the input layer changes and the checkpoint cannot be reused.

The default schedule trains every square map size from `4x4` through `15x15`. It increases `cookie_distance`, uses a moderate food count, grows cookie source count when enough food exists, and expands the episode horizon as the maps get larger.

Each stage trains for exactly `GLOBAL_UPDATE_CAP` TorchRL-backed MAPPO updates. After each stage finishes, the notebook immediately renders that policy, saves the rollout video, archives it in the vault, and then continues to the next map.

In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def curriculum_food_count(size):
    return 2 + max(0, size - 4)


def curriculum_food_sources(size):
    return min(curriculum_food_count(size), max(2, size // 2))


CURRICULUM_STAGES = [
    {
        "name": f"{size}x{size}",
        "width": size,
        "height": size,
        "food_count": curriculum_food_count(size),
        "food_sources": curriculum_food_sources(size),
        "cookie_distance": min(1 + (size - 4) // 2, size // 2),
        "max_steps": max(48, 4 * size * size),
    }
    for size in range(4, 16)
]

MAX_WIDTH = max(stage["width"] for stage in CURRICULUM_STAGES)
MAX_HEIGHT = max(stage["height"] for stage in CURRICULUM_STAGES)
NUM_ENVS = 16
NUM_STEPS = 80
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS
GLOBAL_UPDATE_CAP = 100
ACTOR_VISION_RADIUS = 2
WRITE_BITS = 1
ROLLOUT_TILE_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

COMMON_ARGS = [
    "--num-envs", str(NUM_ENVS),
    "--num-steps", str(NUM_STEPS),
    "--num-minibatches", "4",
    "--update-epochs", "4",
    "--obs-width", str(MAX_WIDTH),
    "--obs-height", str(MAX_HEIGHT),
    "--actor-vision-radius", str(ACTOR_VISION_RADIUS),
    "--write-bits", str(WRITE_BITS),
    "--num-ants", "1",
    "--random-food",
    "--random-hub",
    "--pickup-bonus", "0.25",
    "--distance-bonus", "0.02",
    "--hidden-size", "128",
    "--seed", "1",
    "--quiet",
]


def render_policy_rollout(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    saved_args = SimpleNamespace(**checkpoint["args"])
    write_bits = getattr(saved_args, "write_bits", WRITE_BITS)
    vision_radius = saved_args.actor_vision_radius

    agent = MAPPOAgent(
        central_obs_dim=checkpoint["central_obs_dim"],
        actor_obs_dim=checkpoint["actor_obs_dim"],
        hidden_size=saved_args.hidden_size,
        write_value_count=write_value_count(write_bits),
    ).to(device)
    agent.load_state_dict(checkpoint["agent_state_dict"])
    agent.eval()

    env = AntByteForagingEnv(
        width=saved_args.width,
        height=saved_args.height,
        num_ants=saved_args.num_ants,
        food_count=saved_args.food_count,
        food_source_count=saved_args.food_sources,
        max_steps=saved_args.max_steps,
        random_food=saved_args.random_food,
        render_mode="rgb_array",
        tile_size=ROLLOUT_TILE_SIZE,
        write_bits=write_bits,
    )

    frames = []
    try:
        obs, info = env.reset(
            seed=saved_args.seed,
            options=build_curriculum_reset_options(saved_args, seed=saved_args.seed),
        )
        frame = env.render()
        if frame is not None:
            frames.append(
                draw_vision_squares(
                    frame,
                    obs,
                    tile_size=ROLLOUT_TILE_SIZE,
                    vision_radius=vision_radius,
                )
            )

        for _ in tqdm(
            range(saved_args.max_steps),
            desc=f"{checkpoint_path.stem} rollout",
            leave=False,
        ):
            obs_batch = {key: value[None, ...] for key, value in obs.items()}
            obs_tensor = obs_to_tensor(obs_batch, device)
            central_obs = build_central_observations(
                obs_tensor,
                food_scale=saved_args.food_count,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            actor_obs = build_actor_observations(
                obs_tensor,
                central_obs,
                food_scale=saved_args.food_count,
                actor_vision_radius=saved_args.actor_vision_radius,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )

            with torch.no_grad():
                joint_actions, _, _, _ = agent.get_action_and_value(
                    actor_obs,
                    central_obs,
                    deterministic=False,
                )

            env_action = flatten_agent_actions(joint_actions).cpu().numpy()[0]
            obs, reward, terminated, truncated, info = env.step(env_action)
            frame = env.render()
            if frame is not None:
                frames.append(
                    draw_vision_squares(
                        frame,
                        obs,
                        tile_size=ROLLOUT_TILE_SIZE,
                        vision_radius=vision_radius,
                    )
                )
            if terminated or truncated:
                break
    finally:
        env.close()

    if not frames:
        raise RuntimeError(f"No frames were rendered for {checkpoint_path}.")

    video_path = checkpoint_path.with_name(f"{checkpoint_path.stem}_rollout.mp4")
    imageio.mimsave(video_path, frames, fps=AntByteForagingEnv.metadata["render_fps"])
    return video_path


stage_metrics = []
stage_video_paths = []
stage_vault_entries = []
previous_checkpoint = None

for stage_index, stage in enumerate(CURRICULUM_STAGES, start=1):
    print(f"Training stage {stage_index}/{len(CURRICULUM_STAGES)}: {stage['name']}")
    checkpoint_path = CHECKPOINT_DIR / f"mappo_forage_stage1_{stage['name']}.pt"
    load_checkpoint = previous_checkpoint

    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{stage['name']}",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )
    for update_index in update_iterator:
        train_args = [
            *COMMON_ARGS,
            "--total-timesteps", str(UPDATE_TIMESTEPS),
            "--width", str(stage["width"]),
            "--height", str(stage["height"]),
            "--food-count", str(stage["food_count"]),
            "--food-sources", str(stage["food_sources"]),
            "--cookie-distance", str(stage["cookie_distance"]),
            "--max-steps", str(stage["max_steps"]),
            "--save-model", str(checkpoint_path),
        ]
        if load_checkpoint is not None:
            train_args.extend(["--load-model", str(load_checkpoint)])

        train_metrics = main(train_args)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        metrics_row = {
            **stage,
            **train_metrics,
            "stage_update": update_index,
            "global_update_cap": GLOBAL_UPDATE_CAP,
            "checkpoint": str(checkpoint_path),
        }
        stage_metrics.append(metrics_row)

        load_checkpoint = checkpoint_path

    video_path = render_policy_rollout(checkpoint_path)
    vault_entry_path = create_vault_entry(
        vault_dir=PROJECT_ROOT / "vault",
        title=f"MAPPO policy rollout {stage['name']}",
        description=f"Rollout video after training stage {stage['name']} for {GLOBAL_UPDATE_CAP} MAPPO updates.",
        assets=[video_path],
        metadata={
            "stage": stage,
            "checkpoint_path": str(checkpoint_path),
            "video_path": str(video_path),
            "actor_vision_radius": ACTOR_VISION_RADIUS,
            "write_bits": WRITE_BITS,
            "global_update_cap": GLOBAL_UPDATE_CAP,
        },
    )
    stage_video_paths.append(video_path)
    stage_vault_entries.append(vault_entry_path)

    print(f"Saved rollout video to {video_path}")
    print(f"Archived rollout in {vault_entry_path}")

    previous_checkpoint = checkpoint_path

FINAL_CHECKPOINT_PATH = previous_checkpoint
{
    "stage_metrics": stage_metrics,
    "stage_video_paths": stage_video_paths, 
    "stage_vault_entries": stage_vault_entries,
}

## Optional Re-Render Rollouts

The training cell already renders and archives each stage immediately after its 500 updates. Run this optional cell only if you want to regenerate videos from existing checkpoints.

In [ ]:
from types import SimpleNamespace

import imageio.v2 as imageio
import torch

from ant_byte_env import AntByteForagingEnv
from train_mappo import (
    MAPPOAgent,
    build_actor_observations,
    build_central_observations,
    build_curriculum_reset_options,
    draw_vision_squares,
    flatten_agent_actions,
    obs_to_tensor,
    write_value_count,
)
from ant_byte_env.vault import create_vault_entry

ROLLOUT_TILE_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def render_policy_rollout(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    saved_args = SimpleNamespace(**checkpoint["args"])
    write_bits = getattr(saved_args, "write_bits", globals().get("WRITE_BITS", 1))
    vision_radius = saved_args.actor_vision_radius

    agent = MAPPOAgent(
        central_obs_dim=checkpoint["central_obs_dim"],
        actor_obs_dim=checkpoint["actor_obs_dim"],
        hidden_size=saved_args.hidden_size,
        write_value_count=write_value_count(write_bits),
    ).to(device)
    agent.load_state_dict(checkpoint["agent_state_dict"])
    agent.eval()

    env = AntByteForagingEnv(
        width=saved_args.width,
        height=saved_args.height,
        num_ants=saved_args.num_ants,
        food_count=saved_args.food_count,
        food_source_count=saved_args.food_sources,
        max_steps=saved_args.max_steps,
        random_food=saved_args.random_food,
        render_mode="rgb_array",
        tile_size=ROLLOUT_TILE_SIZE,
        write_bits=write_bits,
    )

    frames = []
    try:
        obs, info = env.reset(
            seed=saved_args.seed,
            options=build_curriculum_reset_options(saved_args, seed=saved_args.seed),
        )
        frame = env.render()
        if frame is not None:
            frames.append(
                draw_vision_squares(
                    frame,
                    obs,
                    tile_size=ROLLOUT_TILE_SIZE,
                    vision_radius=vision_radius,
                )
            )

        for _ in tqdm(
            range(saved_args.max_steps),
            desc=f"{checkpoint_path.stem} rollout",
            leave=False,
        ):
            obs_batch = {key: value[None, ...] for key, value in obs.items()}
            obs_tensor = obs_to_tensor(obs_batch, device)
            central_obs = build_central_observations(
                obs_tensor,
                food_scale=saved_args.food_count,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            actor_obs = build_actor_observations(
                obs_tensor,
                central_obs,
                food_scale=saved_args.food_count,
                actor_vision_radius=saved_args.actor_vision_radius,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )

            with torch.no_grad():
                joint_actions, _, _, _ = agent.get_action_and_value(
                    actor_obs,
                    central_obs,
                    deterministic=False,
                )

            env_action = flatten_agent_actions(joint_actions).cpu().numpy()[0]
            obs, reward, terminated, truncated, info = env.step(env_action)
            frame = env.render()
            if frame is not None:
                frames.append(
                    draw_vision_squares(
                        frame,
                        obs,
                        tile_size=ROLLOUT_TILE_SIZE,
                        vision_radius=vision_radius,
                    )
                )
            if terminated or truncated:
                break
    finally:
        env.close()

    if not frames:
        raise RuntimeError(f"No frames were rendered for {checkpoint_path}.")

    video_path = checkpoint_path.with_name(f"{checkpoint_path.stem}_rollout.mp4")
    imageio.mimsave(video_path, frames, fps=AntByteForagingEnv.metadata["render_fps"])
    return video_path


policy_checkpoint_paths = [
    CHECKPOINT_DIR / f"mappo_forage_stage1_{stage['name']}.pt"
    for stage in CURRICULUM_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing stage policies before rendering:\n{missing}")

video_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=PROJECT_ROOT / "vault",
    title="MAPPO curriculum policy rollouts",
    description="Rollout videos for each saved MAPPO curriculum stage policy.",
    assets=video_paths,
    metadata={
        "stages": [stage["name"] for stage in CURRICULUM_STAGES],
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "actor_vision_radius": ACTOR_VISION_RADIUS,
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "video_paths": video_paths,
    "vault_entry_path": vault_entry_path,
}